# 2 Channel Readout Firmware 512 MHZ bw - Development Notebook

In [1]:
from pynq import Overlay
from pynq import MMIO
import xrfclk
import xrfdc
import struct
from time import sleep
from matplotlib import pyplot as plt
import numpy as np
from scipy import signal
import ipaddress

In [2]:
# FIRMWARE UPLOAD
firmware = Overlay("mkidreadout_dualchan_ethcontrol0.5.xsa",ignore_version=True)
# firmware = Overlay("burrito_wrapper.xsa")

In [3]:
# INITIALIZING PLLs
xrfclk.set_all_ref_clks(409.60)

In [4]:
# Configure Ethernet
ethRegMap={
    "srcip": 0x10,
    "dstip": 0x18,
    "dstmaclsb": 0x20,
    "dstmacmsb": 0x24,
    "ports": 0x2c,
    "timemsb": 0x34, # Not Implemented
    "timelsb": 0x38  # Not Implemented
}

In [5]:
ethernetPort1 = firmware.ethWrapPort0.EthernetControl_0
ethernetPort2 = firmware.ethWrapPort1.EthernetControl_0


In [6]:
ethernetPort1.write(ethRegMap['srcip'], int(ipaddress.IPv4Address(
                       "192.168.3.50"
                   )))
ethernetPort1.write(ethRegMap['dstip'], int(ipaddress.IPv4Address(
                       "192.168.3.40"
                   )))
ethernetPort1.write(ethRegMap['dstmacmsb'],
                   0x803F
                   ) 
ethernetPort1.write(ethRegMap['dstmaclsb'],
                   0x5D092BB0
                   ) 

sourceport = 4096
destport = 4096
ethernetPort1.write(ethRegMap['ports'],
                   (destport<<16)|sourceport) 

In [35]:
# phi = np.random.uniform(-np.pi, np.pi, 1000)
def generate_wave_ddr4(freq_list):    
    fs = 256e6 
    lut_len = 2**20
    fft_len = 1024
    k = np.int64(np.round(freq_list/(fs/lut_len)))
    freq_actual = k*(fs/lut_len)
    X = np.zeros(lut_len,dtype='complex')
    phi = np.random.uniform(-np.pi, np.pi, np.size(freq_list))
    X[k] = np.exp(-1j*phi)
    x = np.fft.ifft(X) * lut_len/np.sqrt(2)
    bin_num = np.int64(np.round(freq_actual / (fs / fft_len)))
    f_beat = (bin_num)*fs/fft_len - (freq_actual)
    dphi0 = f_beat/(fs/fft_len)*2**16
    if np.size(dphi0) > 1:
        dphi = np.concatenate((dphi0, np.zeros(fft_len - np.size(dphi0))))
    else:
        z = np.zeros(fft_len)
        z[0] = dphi0
        dphi = z
    return x, dphi, freq_actual

def norm_wave(wave, max_amp=2**14-1):
    norm = np.max(np.abs(wave))
    if norm == 0:
        return wave.real, wave.imag
    wave_real = ((wave.real/norm)*max_amp).astype("int16")
    wave_imag = ((wave.imag/norm)*max_amp).astype("int16")
    return wave_real, wave_imag

In [36]:
def load_bin_list(chan, freq_list):
    fs = 512e6
    fft_len = 1024
    lut_len = 2**20
    k = np.int64(np.round(-freq_list/(fs/lut_len)))
    freq_actual = k*(fs/lut_len)
    bin_list = np.int64(np.round(freq_actual / (fs / fft_len)))
    pos_bin_idx = np.where(bin_list > 0)
    if np.size(pos_bin_idx) > 0:
        bin_list[pos_bin_idx] = 1024 - bin_list[pos_bin_idx]
    bin_list = np.abs(bin_list)
    # DSP REGS
    if chan == 1:
        dsp_regs = firmware.chan1.dsp_regs_0
    elif chan == 2:
        dsp_regs = firmware.chan2.dsp_regs_0
    else:
        return "Does not compute"
    # 0x00 -  fft_shift[9 downto 0], load_bins[22 downto 12], lut_counter_rst[11 downto 11] 
    # 0x04 -  bin_num[9 downto 0]
    # 0x08 -  accum_len[23 downto 0], accum_rst[24 downto 24], sync_in[26 downto 26] (start dac)
    # 0x0c -  dds_shift[8 downto 0]
    
    # initialization 
    sync_in = 2**26
    accum_rst = 2**24  # (active low)
    accum_length = (2**19)-1
    ################################################
    # Load DDC bins
    ################################################
    offs=21
    
    # only write tones to bin list
    for addr in range(1024):
        if addr<(np.size(bin_list)):
            #print("addr = {}, bin# = {}".format(addr, bin_list[addr]))
            dsp_regs.write(0x04,int(bin_list[addr]))
            dsp_regs.write(0x00, ((addr<<1)+1)<<12)
            dsp_regs.write(0x00, 0)
            print(f"set dsp regs addr = {addr} int(bin_list[addr])={int(bin_list[addr])}, ((addr<<1)+1)<<12={((addr<<1))}")
        else:
            dsp_regs.write(0x04, 0)
            dsp_regs.write(0x00, ((addr<<1)+1)<<12)
            dsp_regs.write(0x00, 0)
    return

def reset_accum_and_sync(chan, freqs):
    if chan == 1:
        dsp_regs = firmware.chan1.dsp_regs_0
        dsp_regs.write(0x0c,181)
    elif chan == 2:
        dsp_regs = firmware.chan2.dsp_regs_0
        dsp_regs.write(0x0c,181)
    else:
        return "Does not compute"
    # dsp_regs bitfield map
    # 0x00 -  fft_shift[9 downto 0], load_bins[22 downto 12], lut_counter_rst[11 downto 11] 
    # 0x04 -  bin_num[9 downto 0]
    # 0x08 -  accum_len[23 downto 0], accum_rst[24 downto 24], sync_in[26 downto 26] (start dac)
    # 0x0c -  dds_shift[8 downto 0]
    # initialization
    sync_in = 2**26
    accum_rst = 2**24  # (active rising edge)
    accum_length = (2**19)-1#(2**19)-1 # (2**18)-1
    
    fft_shift=0
    if len(freqs)<400:
        fft_shift = 511 #2**9-1
    else:
        fft_shift = (2**9)-1
    dsp_regs.write(0x00, fft_shift) # set fft shift
    ########################
    dsp_regs.write(0x08, accum_length | sync_in)
    sleep(0.5)
    dsp_regs.write(0x08, accum_length | accum_rst | sync_in)
#     dsp_regs.write(0x0c,272)#260)
    return

def load_ddr4(chan, wave_real, wave_imag, dphi):
    if chan == 1:
        base_addr_dphis = 0xa004c000
    elif chan == 2:
        base_addr_dphis = 0xa0040000
    else:
        return "Does not compute"
    
    # write dphi to bram
    dphi_16b = dphi.astype("uint16")
    dphi_stacked = ((np.uint32(dphi_16b[1::2]) << 16) + dphi_16b[0::2]).astype("uint32")
    mem_size = 512 * 4 # 32 bit address slots
    mmio_bram_phis = MMIO(base_addr_dphis, mem_size)
    mmio_bram_phis.array[0:512] = dphi_stacked[0:512] # the [0:512] indexing is necessary on .array
    
    # slice waveform for uploading to ddr4
    Q0, Q1, Q2, Q3 = wave_real[0::4], wave_real[1::4], wave_real[2::4], wave_real[3::4]
    I0, I1, I2, I3 = wave_imag[0::4], wave_imag[1::4], wave_imag[2::4], wave_imag[3::4]
    data0 = ((np.int32(I1) << 16) + I0).astype("int32")
    data1 = ((np.int32(Q1) << 16) + Q0).astype("int32")
    data2 = ((np.int32(I3) << 16) + I2).astype("int32")
    data3 = ((np.int32(Q3) << 16) + Q2).astype("int32")
    # write waveform to DDR4 memory
    ddr4mux = firmware.axi_ddr4_mux
    ddr4mux.write(8,0) # set read valid 
    ddr4mux.write(0,0) # mux switch
    ddr4 = firmware.ddr4_0
    base_addr_ddr4 = 0x4_0000_0000 #0x5_0000_0000
    depth_ddr4 = 2**32
    mmio_ddr4 = MMIO(base_addr_ddr4, depth_ddr4)
    mmio_ddr4.array[0:4194304][0 + (chan-1)*4::16] = data0
    mmio_ddr4.array[0:4194304][1 + (chan-1)*4::16] = data1
    mmio_ddr4.array[0:4194304][2 + (chan-1)*4::16] = data2
    mmio_ddr4.array[0:4194304][3 + (chan-1)*4::16] = data3
    ddr4mux.write(8,1) # set read valid 
    ddr4mux.write(0,1) # mux switch
    return

# capture data from ADC
def get_snap_data(chan, mux_sel):
    # WIDE BRAM
    if chan==1:
        axi_wide = firmware.chan1.axi_wide_ctrl# 0x0 max count, 0x8 capture rising edge trigger
        base_addr_wide = 0x00_A007_0000
    elif chan==2:
        axi_wide = firmware.chan2.axi_wide_ctrl
        base_addr_wide = 0x00_B000_0000
    elif chan==3:
        axi_wide = firmware.chan3.axi_wide_ctrl
        base_addr_wide = 0x00_B000_8000
    elif chan==4:
        axi_wide = firmware.chan4.axi_wide_ctrl
        base_addr_wide = 0x00_8200_0000
    else:
        return "Does not compute"
    max_count = 32768
    axi_wide.write(0x08, mux_sel<<1) # mux select 0-adc, 1-pfb, 2-ddc, 3-accum
    axi_wide.write(0x00, max_count - 16) # -4 to account for extra delay in write counter state machine
    axi_wide.write(0x08, mux_sel<<1 | 0)
    axi_wide.write(0x08, mux_sel<<1 | 1)
    axi_wide.write(0x08, mux_sel<<1 | 0)
    mmio_wide_bram = MMIO(base_addr_wide,max_count)
    wide_data = mmio_wide_bram.array[0:8192]# max/4, bram depth*word_bits/32bits
    if mux_sel==0:
        #adc parsing
        up0, lw0 = np.int16(wide_data[0::4] >> 16), np.int16(wide_data[0::4] & 0x0000ffff)
        up1, lw1 = np.int16(wide_data[1::4] >> 16), np.int16(wide_data[1::4] & 0x0000ffff)
        I = np.zeros(4096)
        Q = np.zeros(4096)
        Q[0::2] = lw0
        Q[1::2] = up0
        I[0::2] = lw1
        I[1::2] = up1
    elif mux_sel==1:
        # pfb
        chunk0 = (np.uint64(wide_data[1::4]) << np.uint64(32)) + np.uint64(wide_data[0::4])
        chunk1 = (np.uint64(wide_data[2::4]) << np.uint64(32)) + np.uint64(wide_data[1::4])
        q0 = np.int64((chunk0 & 0x000000000003ffff)<<np.uint64(46))/2**32
        i0 = np.int64(((chunk0>>18) & 0x000000000003ffff)<<np.uint64(46))/2**32
        q1 = np.int64(((chunk1>>4)  & 0x000000000003ffff)<<np.uint64(46))/2**32
        i1 = np.int64(((chunk1>>22)  & 0x000000000003ffff)<<np.uint64(46))/2**32
        I = np.zeros(4096)
        Q = np.zeros(4096)
        Q[0::2] = q0/2**14
        Q[1::2] = q1/2**14
        I[0::2] = i0/2**14
        I[1::2] = i1/2**14
    elif mux_sel==2:
        # ddc
        chunk0 = (np.uint64(wide_data[1::4]) << np.uint64(32)) + np.uint64(wide_data[0::4])
        chunk1 = (np.uint64(wide_data[2::4]) << np.uint64(32)) + np.uint64(wide_data[1::4])
        q0 = np.int64((chunk0 & 0x00000000000fffff)<<np.uint64(45))/2**32
        i0 = np.int64(((chunk0>>19) & 0x00000000000fffff)<<np.uint64(45))/2**32
        q1 = np.int64(((chunk1>>6)  & 0x00000000000fffff)<<np.uint64(45))/2**32
        i1 = np.int64(((chunk1>>25)  & 0x00000000000fffff)<<np.uint64(45))/2**32
        I = np.zeros(4096)
        Q = np.zeros(4096)
        Q[0::2] = q0/2**13
        Q[1::2] = q1/2**13
        I[0::2] = i0/2**13
        I[1::2] = i1/2**13
    elif mux_sel==3:
        # accum
        q0 = (np.int32(wide_data[1::4])).astype("float")
        i0 = (np.int32(wide_data[0::4])).astype("float")
        q1 = (np.int32(wide_data[3::4])).astype("float")
        i1 = (np.int32(wide_data[2::4])).astype("float")
        I = np.zeros(4096)
        Q = np.zeros(4096)
        Q[0::2] = q0
        Q[1::2] = q1
        I[0::2] = i0
        I[1::2] = i1    
    return I, Q

In [37]:
# capture data from ADC
def get_snap_data2(chan, mux_sel):
    # WIDE BRAM
    if chan==1:
        axi_wide = firmware.chan1.axi_wide_ctrl# 0x0 max count, 0x8 capture rising edge trigger
        base_addr_wide = 0x00_A007_0000
    elif chan==2:
        axi_wide = firmware.chan2.axi_wide_ctrl
        base_addr_wide = 0x00_B000_0000
    elif chan==3:
        axi_wide = firmware.chan3.axi_wide_ctrl
        base_addr_wide = 0x00_B000_8000
    elif chan==4:
        axi_wide = firmware.chan4.axi_wide_ctrl
        base_addr_wide = 0x00_8200_0000
    else:
        return "Does not compute"
    max_count = 32768
    axi_wide.write(0x08, mux_sel<<1) # mux select 0-adc, 1-pfb, 2-ddc, 3-accum
    axi_wide.write(0x00, max_count - 16) # -4 to account for extra delay in write counter state machine
    axi_wide.write(0x08, mux_sel<<1 | 0)
    axi_wide.write(0x08, mux_sel<<1 | 1)
    axi_wide.write(0x08, mux_sel<<1 | 0)
    sleep(0.01)
    axi_wide.write(0x08, mux_sel<<1 | 0)
    axi_wide.write(0x08, mux_sel<<1 | 1)
    axi_wide.write(0x08, mux_sel<<1 | 0)
    mmio_wide_bram = MMIO(base_addr_wide,max_count)
    wide_data = mmio_wide_bram.array[0:8192]# max/4, bram depth*word_bits/32bits
    if mux_sel==0:
        #adc parsing
        up0, lw0 = np.int16(wide_data[0::4] >> 16), np.int16(wide_data[0::4] & 0x0000ffff)
        up1, lw1 = np.int16(wide_data[1::4] >> 16), np.int16(wide_data[1::4] & 0x0000ffff)
        I = np.zeros(4096)
        Q = np.zeros(4096)
        Q[0::2] = lw0
        Q[1::2] = up0
        I[0::2] = lw1
        I[1::2] = up1
    elif mux_sel==1:
        # pfb
        chunk0 = (np.uint64(wide_data[1::4]) << np.uint64(32)) + np.uint64(wide_data[0::4])
        chunk1 = (np.uint64(wide_data[2::4]) << np.uint64(32)) + np.uint64(wide_data[1::4])
        q0 = np.int64((chunk0 & 0x000000000003ffff)<<np.uint64(46))/2**32
        i0 = np.int64(((chunk0>>18) & 0x000000000003ffff)<<np.uint64(46))/2**32
        q1 = np.int64(((chunk1>>4)  & 0x000000000003ffff)<<np.uint64(46))/2**32
        i1 = np.int64(((chunk1>>22)  & 0x000000000003ffff)<<np.uint64(46))/2**32
        I = np.zeros(4096)
        Q = np.zeros(4096)
        Q[0::2] = q0/2**14
        Q[1::2] = q1/2**14
        I[0::2] = i0/2**14
        I[1::2] = i1/2**14
    elif mux_sel==2:
        # ddc
        chunk0 = (np.uint64(wide_data[1::4]) << np.uint64(32)) + np.uint64(wide_data[0::4])
        chunk1 = (np.uint64(wide_data[2::4]) << np.uint64(32)) + np.uint64(wide_data[1::4])
        q0 = np.int64((chunk0 & 0x00000000000fffff)<<np.uint64(45))/2**32
        i0 = np.int64(((chunk0>>19) & 0x00000000000fffff)<<np.uint64(45))/2**32
        q1 = np.int64(((chunk1>>6)  & 0x00000000000fffff)<<np.uint64(45))/2**32
        i1 = np.int64(((chunk1>>25)  & 0x00000000000fffff)<<np.uint64(45))/2**32
        I = np.zeros(4096)
        Q = np.zeros(4096)
        Q[0::2] = q0/2**13
        Q[1::2] = q1/2**13
        I[0::2] = i0/2**13
        I[1::2] = i1/2**13
    elif mux_sel==3:
        # accum
        q0 = (np.int32(wide_data[1::4])).astype("float")
        i0 = (np.int32(wide_data[0::4])).astype("float")
        q1 = (np.int32(wide_data[3::4])).astype("float")
        i1 = (np.int32(wide_data[2::4])).astype("float")
        I = np.zeros(4096)
        Q = np.zeros(4096)
        Q[0::2] = q0
        Q[1::2] = q1
        I[0::2] = i0
        I[1::2] = i1    
    return I, Q

In [38]:
# myarrL = np.arange(-255, -5, 0.5)
# myarrR = np.arange(5, 255, 0.5)
# print("myarrL", myarrL.shape)
# print(myarrL[0:5])
# print("myarrR", myarrR.shape)
# print(myarrR[0:5])

# fr = np.append(myarrL, myarrR)*1e6

In [42]:
#  freqs_requested = np.array([50e6, 75e6, 100e6, 107e6, 209e6])

# freqs_requested = np.array(np.linspace(-245.127e6, 255.002e6, 1000))
freqs_requested = np.array([150e6])

wave, dphi, freq_actual = generate_wave_ddr4(freqs_requested);
wave_real, wave_imag = norm_wave(wave, max_amp=2**15-1)
load_ddr4( 1, wave_real, wave_imag, dphi)
load_bin_list( 1, freq_actual)
reset_accum_and_sync( 1, freq_actual)

set dsp regs addr = 0 int(bin_list[addr])=300, ((addr<<1)+1)<<12=0


In [ ]:







# freqs_requested = np.array((30e6, 100e6, 200e6))
# freqs_requested = np.array(np.linspace(-245.127e6, 255.002e6, 1000))
# freqs_requested = np.array(np.linspace(-250e6, 255.002e6, 200))
wave, dphi, freq_actual = generate_wave_ddr4(freqs_requested);
wave_real, wave_imag = norm_wave(wave, max_amp=2**15-1)
load_ddr4( 2, wave_real, wave_imag, dphi)
load_bin_list( 2, freq_actual)
reset_accum_and_sync( 2, freq_actual)

In [ ]:
CHANNEL = 1

In [ ]:
I, Q = get_snap_data(CHANNEL, 0)
c = slice(3072,4096)
plt.figure(figsize=(14,8))

plt.plot(I[c])
plt.plot(Q[c])
plt.legend(['I', 'Q'])
plt.xlim(0,150)
# I = I[0:1023]
# Q = Q[0:1023]
plt.figure(figsize=(14,8))

               
               
fft_x = np.fft.fftfreq(1024, d=1/512e6)
fft_y = np.fft.fft(I[c] + 1j*Q[c], 1024)
# fft_x = np.fft.fftshift(fft_x)
fft_pow = fft_y.real**2 + fft_y.imag**2

plt.stem(fft_x[0:512]/1e6, fft_pow[0:512])
plt.xlabel("Freq. [MHz]")
plt.show()

plt.figure()
plt.plot(np.arctan2(Q,I))
plt.xlim(0,150)

In [ ]:
#Take Average
n_averages = 100
spectra = np.zeros([n_averages, 1024]) # fftlen
for i in range(10):
    I, Q = get_snap_data(CHANNEL, 0)
    c = slice(3072,4096)
    fft_y = np.fft.fft(I[c] + 1j*Q[c], 1024)
    fft_pow = fft_y.real**2 + fft_y.imag**2
    spectra[i,:] = fft_pow

In [ ]:
# Plot average
meanfft = np.mean(spectra)

plt.figure(figsize=(14,8))
plt.stem(fft_x, fft_pow)
plt.xlabel("Freq. [MHz]")
plt.show()

In [ ]:
I, Q = get_snap_data(CHANNEL, 2)
Iddc_f = I[c].astype("float")/2**13
Qddc_f = Q[c].astype("float")/2**13
IQddc=abs(Iddc_f+1j*Qddc_f)

plt.figure()
plt.title(f"DDC of CH {CHANNEL}",fontsize=16)
plt.plot(Iddc_f,"-+",label='I')
plt.plot(Qddc_f,"-+",label='Q')
plt.xlim((1000,1024))
plt.legend()
plt.show()

In [ ]:
I, Q = get_snap_data(CHANNEL, 1)
plt.figure()
z = I + 1j*Q
plt.plot(abs(z[c]))

In [ ]:
I, Q = get_snap_data(CHANNEL, 3)
plt.figure(figsize=(14,8))
plt.plot(10*np.log(np.sqrt(I[c]**2 + Q[c]**2)))
# plt.stem(abs(z[c]))
# plt.xlim(0,10)

In [ ]:
I, Q = get_snap_data(CHANNEL, 3)
Iacc_f = I/2**10
Qacc_f = Q/2**10
Ia, Qa = np.array(Iacc_f).astype(float), np.array(Qacc_f).astype(float)
IQMAG=np.sqrt(Ia**2+Qa**2)
print(f"{Ia.shape}")

plt.figure(figsize=(8,5))
plt.plot(Ia,label='I')
plt.plot(Qa,label='Q')
plt.legend()
plt.title("ACCUM")
plt.show()

plt.figure(figsize=(12,7))

plt.plot(IQMAG[c][:],"-+")
plt.xlabel('Tone Number',fontsize=16)
plt.ylabel('ADU',fontsize=16)
plt.title(f"IQ MAG of Channel {CHANNEL}",fontsize=16)
# plt.xlim((0,13))?/

plt.show()
np.argmax(IQMAG)

In [ ]:
spectra_t_i = np.zeros((100, 1024))
spectra_t_q = np.zeros((100, 1024))
for i in range(100):
    I, Q = get_snap_data(CHANNEL, 3)
    Iacc_f = I[0:1024]/2**10
    Qacc_f = Q[0:1024]/2**10
    Ia, Qa = np.array(Iacc_f).astype(float), np.array(Qacc_f).astype(float)
    spectra_t_i[i][:] = Ia
    spectra_t_q[i][:] = Qa

In [ ]:
A = spectra_t_i.flatten()
B = spectra_t_q.flatten()

In [ ]:
plt.figure()
plt.plot(A[4::1024])
plt.figure()
plt.plot(B[4::1024])